# 07 — Baseline Model

This notebook establishes a baseline forecasting model for car registrations.

The objective is to create a simple, interpretable model using historical registration data and basic time-based features. This baseline serves as a reference point for evaluating more advanced models developed later.

The model primarily relies on:

- Lagged registration values
- Rolling averages to capture recent trends
- Seasonal features (month-based encoding)

The performance of this baseline model provides a benchmark against which more complex models can be compared.

## Why a Baseline Model?

A baseline model helps answer a key question:

> How much improvement do more complex models actually provide?

By starting with a simple and transparent approach, it becomes easier to measure the value added by additional features, macroeconomic variables, and more advanced algorithms.

In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from functools import reduce
import os

from statsmodels.tsa.stattools import adfuller


In [ ]:
# =========================================================
# Load datasets
# =========================================================
baseline_df = pd.read_parquet("data/baseline_df.parquet")

In [ ]:
display(baseline_df.head(20))

In [ ]:
# =========================================================
#  1. Spliting data into Time-Based Train/Test  
# =========================================================



baseline_df = baseline_df.sort_values("date").copy()

train_df = baseline_df[baseline_df["date"] < "2025-01-01"]
test_df  = baseline_df[baseline_df["date"] >= "2025-01-01"]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
# =========================================================
#  2. Define Target & Features  
# =========================================================


TARGET = "acea_total_registrations"

FEATURES = [col for col in baseline_df.columns 
            if col not in ["date", TARGET]]

In [ ]:
print(FEATURES)

In [ ]:
# =========================================================
#  3. Build a Simple Baseline Model  
# =========================================================

from sklearn.linear_model import LinearRegression

model = LinearRegression()

In [ ]:
# =========================================================
#  4. Backtesting (Expanding Window)
# =========================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

def rolling_backtest(
    df,
    start_date,
    features,
    target,
    model
):
    df = df.sort_values("date").reset_index(drop=True).copy()
    
    # make sure date column is datetime
    df["date"] = pd.to_datetime(df["date"])
    
    # convert start_date to datetime
    start_date = pd.to_datetime(start_date)
    
    predictions = []
    actuals = []
    dates = []
    
    for i in range(len(df)):
        current_date = df.loc[i, "date"]
        
        if current_date < start_date:
            continue
        
        train = df[df["date"] < current_date]
        test = df[df["date"] == current_date]
        
        if len(train) == 0:
            continue
        
        X_train = train[features]
        y_train = train[target]
        
        X_test = test[features]
        y_test = test[target]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        predictions.append(y_pred[0])
        actuals.append(y_test.values[0])
        dates.append(current_date)
    
    results = pd.DataFrame({
        "date": dates,
        "actual": actuals,
        "prediction": predictions
    })
    
    return results

In [ ]:
# =========================================================
#  5. Run Backtesting
# =========================================================

results = rolling_backtest(
    df=baseline_df,
    start_date="2024-01-01",
    features=FEATURES,
    target=TARGET,
    model=model
)

In [ ]:
# =========================================================
#  6. Evaluate Model Performance 
# =========================================================

mae = mean_absolute_error(results["actual"], results["prediction"])
rmse = np.sqrt(mean_squared_error(results["actual"], results["prediction"]))
mape = np.mean(np.abs((results["actual"] - results["prediction"]) / results["actual"])) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
# =========================================================
#  7. Compare with Naive Baseline 
# =========================================================


baseline_df["naive_pred"] = baseline_df[TARGET].shift(1)

naive_df = baseline_df.dropna()

mae_naive = mean_absolute_error(
    naive_df[TARGET],
    naive_df["naive_pred"]
)

mape_naive = np.mean(
    np.abs(
        (naive_df[TARGET] - naive_df["naive_pred"]) / naive_df[TARGET]
    )
) * 100

print(f"Naive MAE: {mae_naive:.2f}")
print(f"Naive_MAPE: {mape_naive:.2f}%")

In [ ]:
# =========================================================
#  8. Creating a final results table 
# =========================================================


results["error"] = results["prediction"] - results["actual"]

results["abs_error"] = np.abs(results["error"])

results["error_pct"] = (
    results["abs_error"] / results["actual"]
) * 100


results_table = results[[
    "date",
    "actual",
    "prediction",
    "error",
    "abs_error",
    "error_pct"
]]

results_table["error_pct"] = results_table["error_pct"].round(2)
results_table["prediction"] = results_table["prediction"].round(0)
results_table["actual"] = results_table["actual"].round(0)

display(results_table)

In [ ]:
# =========================================================
#  9. Visualize Results
# =========================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(results["date"], results["actual"], label="Actual")
plt.plot(results["date"], results["prediction"], label="Prediction")
plt.legend()
plt.title("Baseline Model - Rolling Forecast")
plt.show()

In [ ]:
# =========================================================
#  Extract Model Coefficients & R2
# =========================================================

coef_df_baseline = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": model.coef_
})

coef_df_baseline["abs_coef"] = coef_df_baseline["coefficient"].abs()

# Sort by importance
coef_df_baseline = coef_df_baseline.sort_values("abs_coef", ascending=False)

display(coef_df_baseline)

from sklearn.metrics import r2_score

# Fit model on full dataset (for interpretation only)
model.fit(baseline_df[FEATURES], baseline_df[TARGET])

y_pred_full = model.predict(baseline_df[FEATURES])

r2 = r2_score(baseline_df[TARGET], y_pred_full)

print(f"R² (in-sample): {r2:.4f}")